# 第9回　データの表現：グラフ・散布図
## ―― グラフを選ぶことは、何を主張するかを選ぶことである

情報活用Ⅰ　／　北星学園大学　2026年度後期

**日本語フォントは、下の準備セルで設定済み。** そのまま日本語のラベルが出る。
（Colabには日本語フォントが入っていないので、何もしないと軸ラベルが □□□ になる）

In [ ]:
# 準備：ライブラリと、霊長類376種のデータを読み込む。▶ を押すだけ。
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # 日本語が豆腐（□）にならないようにする

def _build_from_source(keep_missing_code=False):
    """公開データ（PanTHERIA）から、この授業で使う形に組み立て直す。"""
    SRC = "https://esapubs.org/archive/ecol/E090/184/PanTHERIA_1-0_WR05_Aug2008.txt"
    fam = {"Cercopithecidae":"オナガザル科","Cebidae":"オマキザル科","Pitheciidae":"サキ科",
           "Atelidae":"クモザル科","Cheirogaleidae":"コビトキツネザル科","Lemuridae":"キツネザル科",
           "Galagidae":"ガラゴ科","Hylobatidae":"テナガザル科","Indriidae":"インドリ科",
           "Lorisidae":"ロリス科","Lepilemuridae":"イタチキツネザル科","Aotidae":"ヨザル科",
           "Hominidae":"ヒト科","Tarsiidae":"メガネザル科","Daubentoniidae":"アイアイ科"}
    cols = {"MSW05_Binomial":"学名","MSW05_Genus":"属","5-1_AdultBodyMass_g":"体重g",
            "13-1_AdultHeadBodyLen_mm":"頭胴長mm","5-3_NeonateBodyMass_g":"新生児体重g",
            "10-2_SocialGrpSize":"集団サイズ","9-1_GestationLen_d":"妊娠期間日",
            "25-1_WeaningAge_d":"離乳日齢","3-1_AgeatFirstBirth_d":"初産日齢",
            "14-1_InterbirthInterval_d":"出産間隔日","15-1_LitterSize":"一腹産子数",
            "17-1_MaxLongevity_m":"最長寿命月","22-1_HomeRange_km2":"行動圏km2",
            "21-1_PopulationDensity_n/km2":"個体群密度","26-1_GR_Area_km2":"分布域km2",
            "6-2_TrophicLevel":"栄養段階","12-1_HabitatBreadth":"生息環境幅",
            "28-2_Temp_Mean_01degC":"平均気温01","28-1_Precip_Mean_mm":"月降水量mm"}
    src = pd.read_csv(SRC, sep="\t")
    p = src[src["MSW05_Order"] == "Primates"]
    out = p[list(cols)].rename(columns=cols)
    out.insert(1, "科", p["MSW05_Family"].map(fam))
    t = out.pop("平均気温01")
    out["平均気温C"] = np.where(t == -999, -999, (t / 10).round(1))
    out = out.sort_values("学名").reset_index(drop=True)
    return out if keep_missing_code else out.replace(-999, np.nan)

try:
    df = pd.read_csv("https://aonoa68.github.io/joho-katsuyo/data/primates.csv")
except Exception:
    df = _build_from_source()

print("種数:", len(df), " 科数:", df["科"].nunique())
df.head()

---
## 1. グラフの種類と、それが答えている問い

**グラフは飾りではない。問いに対する答えの形である。**

| 問い | グラフ | 使う関数 |
|---|---|---|
| どれが多いか（グループの比較） | 棒グラフ | `plt.bar` |
| どう散らばっているか（1つの項目の分布） | ヒストグラム | `plt.hist` |
| 2つの項目に関係があるか | 散布図 | `plt.scatter` |
| グループごとに関係が違うか | 層別散布図 | `plt.scatter` を色分け |
| グループ間で分布を比べたい | 箱ひげ図 | `plt.boxplot` |

**先に問いを決める。それからグラフを選ぶ。** 逆をやると、きれいだが何も言っていない図ができる。

### 棒グラフ ―― どれが多いか

In [ ]:
counts = df["科"].value_counts()

plt.figure(figsize=(9, 4.5))
plt.bar(counts.index, counts.values, color="#80cbc4", edgecolor="white")
plt.ylabel("種数")
plt.title("科ごとの種数（霊長類376種）")
plt.xticks(rotation=45, ha="right")
for i, v in enumerate(counts.values):
    plt.text(i, v + 1.5, str(v), ha="center", fontsize=9)
plt.tight_layout(); plt.show()

---
## 2. ヒストグラム ―― そして「軸の選び方」という判断

**平均だけでは、分布の形は分からない。** 体重の分布を描いてみる。

In [ ]:
w = df["体重g"].dropna()

plt.figure(figsize=(9, 4))
plt.hist(w, bins=50, color="#80cbc4", edgecolor="white")
plt.axvline(w.mean(), color="#e8503a", lw=2, label=f"平均 {w.mean():,.0f}g")
plt.axvline(w.median(), color="#1565c0", lw=2, ls="--", label=f"中央値 {w.median():,.0f}g")
plt.xlabel("体重（g）"); plt.ylabel("種数")
plt.title("霊長類の体重分布")
plt.legend(); plt.show()

**ほとんどの棒が左端に潰れてしまった。** ゴリラ（149kg）まで横軸を伸ばした結果、31gのネズミキツネザルから3kgのサルまでが、全部いちばん左の1本に押し込まれている。

このグラフは嘘はついていないが、**何も見えない**。ここで軸の取り方を変える。

In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(np.log10(w), bins=40, color="#80cbc4", edgecolor="white")
plt.axvline(np.log10(w.median()), color="#1565c0", lw=2, ls="--",
            label=f"中央値 {w.median():,.0f}g")
plt.xlabel("体重の常用対数　log₁₀(g)")
plt.ylabel("種数")
plt.title("横軸を対数にすると、分布の形が見える")
# 目盛りを「読める単位」に戻す
ticks = [1, 2, 3, 4, 5]
plt.xticks(ticks, ["10g", "100g", "1kg", "10kg", "100kg"])
plt.legend(); plt.show()

from scipy import stats
print(f"そのまま   の歪み（歪度）: {stats.skew(w):>6.2f}   ← 0から遠いほど左右非対称")
print(f"対数にした後の歪み        : {stats.skew(np.log(w)):>6.2f}")

**同じデータである。** 横軸を対数にしただけで、左右対称に近い山が現れた。
歪度は 7.6 から −0.4 へ。

> **軸の取り方は「見せ方の工夫」ではなく、分析上の判断である。**
> 体重・年収・都市人口・企業規模のように**何桁にもまたがる数字**は、対数で見るのが普通。
> ただし **対数軸だと明記すること。** 書かなければ、読む人は差を10分の1に見誤る。

---
## 3. 散布図 ―― 2つの項目に関係があるか

**散布図には相関係数を添える。** 目で見た印象と、数字は食い違うことがある。

「体の大きい霊長類ほど、大きな集団で暮らすのか」を見る。

In [ ]:
e = df.dropna(subset=["体重g", "集団サイズ"])
x, y = np.log10(e["体重g"]), np.log10(e["集団サイズ"])
r = x.corr(y)

plt.figure(figsize=(7, 5.5))
plt.scatter(x, y, alpha=0.5, s=26, color="#1565c0")
plt.xlabel("体重　log₁₀(g)"); plt.ylabel("集団サイズ　log₁₀(頭)")
plt.xticks([1,2,3,4,5], ["10g","100g","1kg","10kg","100kg"])
plt.yticks([0,1,2], ["1頭","10頭","100頭"])
plt.title(f"体重と集団サイズ　r = {r:.2f}　(n={len(e)}種)")
plt.show()

print(f"相関係数 r = {r:.3f}   n = {len(e)}種")
print("※ 相関があっても、原因と結果は決まらない（第10回で扱う）")

**右上がりの関係が見える。** ただし点のばらつきは大きく、体重だけで集団サイズが決まるわけではない。

そして忘れてはいけないのは、**この図には376種のうち集団サイズが記録された種しか写っていない**ということ。記録がない種が、たまたま小さな集団の種に偏っていたら、この関係は変わる。

### 層別散布図 ―― グループごとに関係が違うか

In [ ]:
big = e["科"].value_counts()
big = big[big >= 15].index
sub = e[e["科"].isin(big)]

plt.figure(figsize=(8, 5.5))
for fam, g in sub.groupby("科"):
    rr = np.log10(g["体重g"]).corr(np.log10(g["集団サイズ"]))
    plt.scatter(np.log10(g["体重g"]), np.log10(g["集団サイズ"]),
                alpha=0.6, s=28, label=f"{fam} (n={len(g)}, r={rr:.2f})")

plt.xlabel("体重　log₁₀(g)"); plt.ylabel("集団サイズ　log₁₀(頭)")
plt.xticks([1,2,3,4,5], ["10g","100g","1kg","10kg","100kg"])
plt.yticks([0,1,2], ["1頭","10頭","100頭"])
plt.title("科で分けて見る")
plt.legend(fontsize=9); plt.show()

**分けて見ると、グループごとに向きや強さが違う。**第7回のシンプソンのパラドックスと同じ話が、散布図でも起きる。

---
## 4. 悪いグラフ ―― Y軸を0から始めない

**同じデータである。** 描き方だけを変えて2枚並べる。

In [ ]:
m = (df.dropna(subset=["集団サイズ"])
       .groupby("科")["集団サイズ"].mean())
m = m[m.index.isin(big)].sort_values(ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))

ax[0].bar(m.index, m.values, color="#80cbc4", edgecolor="white")
ax[0].set_ylim(0, m.max() * 1.2)
ax[0].set_ylabel("平均集団サイズ（頭）")
ax[0].set_title("Y軸を0から　── 差が正しく見える")
ax[0].tick_params(axis="x", rotation=45)

ax[1].bar(m.index, m.values, color="#ef9a9a", edgecolor="white")
ax[1].set_ylim(m.min() * 0.97, m.max() * 1.01)      # ← ここだけ変えた
ax[1].set_ylabel("平均集団サイズ（頭）")
ax[1].set_title("Y軸を途中から　── 圧倒的な差に見える")
ax[1].tick_params(axis="x", rotation=45)

plt.tight_layout(); plt.show()

右のグラフは、**嘘をついていない。** 数字は正しく、軸にも目盛りが書いてある。
それでも、読む人は「1位が圧勝」という印象を持って帰る。

> **棒グラフのY軸は0から始める。** 棒の長さで量を表す図だから、途中で切ると長さの比が壊れる。

**第2節の対数軸と、何が違うのか。** どちらも軸をいじっている。違いは1つ。

- **対数軸**：何桁にもまたがるデータを見るための、**分析上の必要**。目盛りに単位を書けば誤読されない
- **Y軸の途中打ち切り**：差を大きく見せる以外の目的がない。**しかも棒の長さの比が壊れる**

軸をいじるときは、**「なぜそうしたか」を1文で言えるか**を自分に問う。言えないならやめる。

---
## 5. 優れた可視化とは何か

悪い例だけを見ると「気をつけよう」で終わる。**何がよい可視化なのか**を2つ挙げる。

### ジョン・スノウのコレラ地図（1854年・ロンドン）

コレラの死亡者を、**ロンドンの地図の上に点で打った。** すると、ブロード街の井戸のまわりに点が集中していることが見えた。井戸が原因だという仮説が、そこから立った。

**グラフの形が、問いの立て方そのものだった例である。** 死者数を表で並べても、この発見はなかった。

### Our World in Data のグラフ

軸・単位・出典・注釈がすべて明示されており、**元データをダウンロードして自分で確かめられる。**

「確かめられる形で見せる」ことが、可視化の要件でもある。
**これは、AIの出力に対してこの授業がとる態度と同じ話である** ―― 過程が残っているか、確かめられるか。

> **今日使ったデータにも出典がある**（このページの一番下）。
> あなたが自分のページに図を載せるときも（第10・11回）、> **出典と、何を数えた数字なのかを必ず添える。**

---
## 6. こういう見せ方もある（紹介）

In [ ]:
# 箱ひげ図：グループごとの分布を並べて比べる
groups, labels = [], []
for fam, g in df[df["科"].isin(big)].dropna(subset=["体重g"]).groupby("科"):
    groups.append(np.log10(g["体重g"].values)); labels.append(fam)

plt.figure(figsize=(9, 4.5))
plt.boxplot(groups, patch_artist=True,
            boxprops=dict(facecolor="#b2dfdb"), medianprops=dict(color="#e8503a", lw=2))
plt.xticks(range(1, len(labels) + 1), labels, rotation=45, ha="right")
plt.yticks([1,2,3,4,5], ["10g","100g","1kg","10kg","100kg"])
plt.ylabel("体重（対数目盛り）")
plt.title("科ごとの体重分布（箱＝真ん中の50%、線＝中央値）")
plt.tight_layout(); plt.show()

In [ ]:
# ヒートマップ：数値どうしの関係を一覧する
cols = ["体重g", "頭胴長mm", "集団サイズ", "妊娠期間日", "最長寿命月", "行動圏km2"]
corr = np.log10(df[cols].where(df[cols] > 0)).corr()

plt.figure(figsize=(7, 6))
plt.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(label="相関係数（対数変換後）")
plt.xticks(range(len(cols)), cols, rotation=45, ha="right")
plt.yticks(range(len(cols)), cols)
for i in range(len(cols)):
    for j in range(len(cols)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)
plt.title("項目どうしの相関")
plt.tight_layout(); plt.show()

> **この表の各マスは、それぞれ違う種数から計算されている。**
> 妊娠期間と最長寿命の両方が記録された種は、376種のうちごく一部しかない。
> 見やすい表ほど、その裏にある件数の差が見えなくなる。

---
## 7. 余談 ―― あなたはすでにコマンドを使っている

Colabのセルの先頭に `!` をつけると、**コンピュータに直接命令できる。**

In [ ]:
!ls

いま出てきたのは、**このノートブックが動いている場所にあるファイルの一覧**である。

普段は、フォルダのアイコンをダブルクリックして中身を見る。あれと同じことを、文字で命令した。
前者を **GUI**（画面で操作する）、後者を **CLI**（文字で命令する）という。

第6回で `pd.read_csv("ファイル名")` と書いたとき、あなたはすでに**「どこに何というファイルがあるか」を文字で指定していた。** CLIの世界に片足を入れている。

画面の裏側には、必ずファイルの実体がある。**アイコンは、その見え方の一つでしかない。**

---
## 8. 画像に書き出す（次回、ページに貼る）

第10回で、成果物のページにこのグラフを貼る。**いま画像として書き出しておく。**

In [ ]:
plt.figure(figsize=(8, 5.5))
for fam, g in sub.groupby("科"):
    plt.scatter(np.log10(g["体重g"]), np.log10(g["集団サイズ"]), alpha=0.6, s=28, label=fam)
plt.xlabel("体重　log₁₀(g)"); plt.ylabel("集団サイズ　log₁₀(頭)")
plt.xticks([1,2,3,4,5], ["10g","100g","1kg","10kg","100kg"])
plt.yticks([0,1,2], ["1頭","10頭","100頭"])
plt.title("体重と集団サイズ（科別）")
plt.legend(fontsize=9)

# ← これが書き出しの行。dpi=150 くらいにするとページで粗く見えない
plt.savefig("graph1.png", dpi=150, bbox_inches="tight")
plt.show()

print("graph1.png を書き出した。左の📁からダウンロードできる。")

---
## 9. 自分の調査データでやる

In [ ]:
# ここから先は「自分の調査データ」でやる。
# Google Forms の回答 → スプレッドシート → ファイル → ダウンロード → CSV で書き出したものを使う。
#
# 左のフォルダアイコン（📁）にCSVをドラッグしてから、ファイル名を書き換えて実行する。
# ※ 数字を手で打ち直さないこと。転記した瞬間に、それは元データではなくなる。

# mydf = pd.read_csv("自分のファイル名.csv")
# mydf.head()

In [ ]:
# グラフ1枚目（列名を書き換える）
# plt.figure(figsize=(7, 4))
# plt.hist(mydf["数値の列"].dropna(), bins=15, color="#80cbc4", edgecolor="white")
# plt.xlabel("　"); plt.ylabel("件数"); plt.title("　")
# plt.savefig("mygraph1.png", dpi=150, bbox_inches="tight")
# plt.show()

---
## 10. 卒業 ―― グラフ選択ドリル

**データの種類を見て、グラフを即答できるか。**

👉 **[グラフ選択ドリルを開く](https://aonoa68.github.io/joho-katsuyo/graph-drill.html)**

「この問いに答えるには、どのグラフか」を繰り返す。手が覚えるのではなく、**問いとグラフの対応**が身につけば、道具が変わっても選べる。

余裕があれば、**自分が作ったグラフを1枚選び、キャプションを1文で書く**練習もしておく。キャプションが書けないグラフは、まだ何も主張していない。

---
## 課題9（8点）

**このノートブック** ＋ **グラフ2枚（うち1枚は層別）** ＋ **各グラフの読み取り**。

- [ ] グラフ1枚目。軸ラベルとタイトルを必ず入れる
- [ ] グラフ2枚目は**層別**（何かで色分け・グループ分けする）
- [ ] 散布図を描いたなら、**相関係数と件数（n）を添える**
- [ ] 各グラフについて、読み取ったことを1〜2文
- [ ] **画像を書き出す**（`savefig`）。次回のページで使う

**軸ラベルのないグラフは減点する。** 何の数字かが分からない図は、根拠にならない。
**軸をいじった場合は、その理由を1文で書く。**

提出期限：次回授業の開始まで（遅れた場合は50%）

---

!!! quote "このデータの出典"
    Jones, K.E. et al. (2009) PanTHERIA: a species-level database of life history,
    ecology, and geography of extant and recently extinct mammals.
    *Ecology* 90(9): 2648. Ecological Archives E090-184.